In [1]:
import csv
import warnings
from io import StringIO
from pathlib import Path

import pandas as pd



def parse_patient_dates(values: pd.Series) -> pd.Series:
    """Parse Spanish export dates and ISO dates without relying on system locale."""
    months = {
        "enero": "01", "febrero": "02", "marzo": "03",
        "abril": "04", "mayo": "05", "junio": "06",
        "julio": "07", "agosto": "08", "septiembre": "09",
        "setiembre": "09", "octubre": "10", "noviembre": "11",
        "diciembre": "12",
    }
    text = values.astype("string").str.strip().str.lower()
    text = text.str.replace("\u00e2", "", regex=False)
    text = text.str.replace(r"\s+", " ", regex=True)
    text = text.str.replace(r"([ap])\.\s*m\.", r"\1m", regex=True)
    for month, number in months.items():
        text = text.str.replace(month, number, regex=False)

    parsed = pd.Series(pd.NaT, index=values.index, dtype="datetime64[ns]")
    for date_format in (
        "%m %d, %Y, %I:%M %p", "%m %d, %Y, %I:%M:%S %p",
        "%m %d, %Y", "%Y-%m-%d", "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M:%S.%f",
        "%Y-%m-%dT%H:%M:%S.%f",
    ):
        missing = parsed.isna() & text.notna() & text.ne("")
        parsed.loc[missing] = pd.to_datetime(
            text.loc[missing], format=date_format, errors="coerce"
        ).astype("datetime64[ns]")
    invalid = text.notna() & text.ne("") & parsed.isna()
    if invalid.any():
        warnings.warn(
            f"{values.name}: {invalid.sum()} fechas no interpretables convertidas a NaT.",
            stacklevel=2,
        )
    return parsed


# Edad en años cumplidos a la fecha del extracto.
def calculate_age(birth_dates: pd.Series, reference_date="2026-11-05") -> pd.Series:
    reference = pd.Timestamp(reference_date).normalize()
    birthday_pending = (birth_dates.dt.month > reference.month) | (
        (birth_dates.dt.month == reference.month)
        & (birth_dates.dt.day > reference.day)
    )
    age = (
        reference.year - birth_dates.dt.year - birthday_pending.astype("Int64")
    ).astype("Int64")
    return age.mask(birth_dates.isna() | (birth_dates > reference))


def load_patient_csv(
    path: str | Path, *, reference_date="2026-11-05"
) -> pd.DataFrame:
    """Load the wrapped cohort export without modifying the source CSV."""
    path = Path(path)
    with path.open("r", encoding="utf-8-sig", newline="") as csv_file:
        lines = csv_file.read().splitlines()

    if not lines:
        raise ValueError(f"The CSV file is empty: {path}")

    header = next(csv.reader([lines[0]]))
    if header[:2] != ["patient_ref", "birth_date"]:
        raise ValueError("Expected patient_ref and birth_date as the first two columns.")

    records = []
    for line_number, raw_line in enumerate(lines[1:], start=2):
        if not raw_line.strip():
            continue

        stripped_line = raw_line.strip()
        fields = next(csv.reader([raw_line]))
        if (
            len(fields) == 1
            and stripped_line.startswith('"')
            and stripped_line.endswith('"')
        ):
            # The export wraps each complete row in quotes and doubles its inner quotes.
            unwrapped_line = stripped_line[1:-1].replace('""', '"')
            fields = next(csv.reader([unwrapped_line]))

        extra_fields = len(fields) - len(header)
        if extra_fields < 0:
            raise ValueError(
                f"Row {line_number} has fewer fields than the {len(header)}-column header."
            )
        if extra_fields:
            # Commas in patient_ref were not escaped, so join the split ID pieces back.
            fields = [",".join(fields[: extra_fields + 1]), *fields[extra_fields + 1 :]]

        if len(fields) != len(header):
            raise ValueError(
                f"Row {line_number} has {len(fields)} fields after repair; "
                f"expected {len(header)}."
            )
        records.append(fields)

    repaired_csv = StringIO()
    writer = csv.writer(repaired_csv)
    writer.writerow(header)
    writer.writerows(records)
    repaired_csv.seek(0)

    frame = pd.read_csv(repaired_csv)
    date_columns = [
        column for column in frame.columns
        if column.startswith("fecha_") or column.endswith("_date")
    ]
    for column in date_columns:
        frame[column] = parse_patient_dates(frame[column])
    # Sustituir la fecha de nacimiento por la edad en la misma posicion.
    age = calculate_age(frame["birth_date"], reference_date)
    frame = frame.drop(columns="age", errors="ignore")
    position = frame.columns.get_loc("birth_date")
    frame.pop("birth_date")
    frame.insert(position, "age", age)
    return frame


df = load_patient_csv("./private/query_result_2026-11-05T08_34_59.csv")
print(f"Loaded {len(df)} rows and {len(df.columns)} columns.")

# Export the repaired/formatted table as a standard CSV.
formatted_csv_path = Path("./private/query_result_formatted.csv")
df.to_csv(formatted_csv_path, index=False, encoding="utf-8-sig")
print(f"Formatted CSV saved to: {formatted_csv_path}")

ModuleNotFoundError: No module named 'pandas'

In [9]:
# A data dictionary: one row per source column, with a readable name and meaning.
column_descriptions = {
    "patient_ref": ("Referencia del paciente", "Identificador del registro del paciente. Es un dato sensible y debe utilizarse solo en sistemas autorizados.", "Demografía", 3),
    "age": ('Edad', 'Edad en años cumplidos a la fecha de referencia (por defecto, 2026-11-05).', 'Demografía', 3),
    "lab_sap_ref": ("Código de la prueba de laboratorio", "Código SAP utilizado para identificar o rastrear la prueba de laboratorio.", "Laboratorio", 4),
    "lab_descr": ("Nombre de la prueba de laboratorio", "Nombre del analito medido; los registros actuales indican Apolipoproteína A1.", "Laboratorio", 4),
    "ultima_lpa": ("Último valor de laboratorio registrado", "Resultado numérico de laboratorio. Aunque el nombre del campo sugiere Lp(a), compruebe el analito en lab_descr antes de interpretarlo.", "Laboratorio", 4),
    "lpa_units": ("Unidades del valor de laboratorio", "Unidades registradas para el resultado de laboratorio, como mg/dL.", "Laboratorio", 4),
    "lpa_extrac_date": ("Fecha de extracción de la muestra", "Posible fecha y hora de extracción de la muestra; debe confirmarse con el sistema de origen.", "Laboratorio", 4),
    "lpa_result_date": ("Fecha de emisión del resultado", "Posible fecha y hora de emisión del resultado de laboratorio; debe confirmarse con el sistema de origen.", "Laboratorio", 4),
    "fecha_ultima_lpa": ("Fecha de la última prueba de Lp(a)", "Fecha asociada a la última prueba de Lp(a); confirme su relación con las fechas de extracción y emisión del resultado.", "Laboratorio", 4),
    "tiene_visita_24m": ("Visita en los últimos 24 meses", "Indica si se registró una visita que cumple los criterios durante el período de 24 meses; confirme la definición del período.", "Seguimiento", 5),
    "fecha_ultima_visita_24m": ("Fecha de la última visita que cumple los criterios", "Fecha de la última visita incluida en el indicador de visitas de los últimos 24 meses.", "Seguimiento", 5),
    "seguimiento_endocrino": ("Seguimiento en endocrinología registrado", "Indica seguimiento registrado en endocrinología; por sí solo no establece un diagnóstico ni una enfermedad activa.", "Seguimiento", 5),
    "fecha_ultima_visita_endocrino": ("Fecha de la última visita de endocrinología", "Fecha de la visita más reciente registrada en endocrinología.", "Seguimiento", 5),
    "seguimiento_cardio": ("Seguimiento en cardiología registrado", "Indica seguimiento registrado en cardiología; por sí solo no establece un diagnóstico ni una enfermedad activa.", "Seguimiento", 5),
    "fecha_ultima_visita_cardio": ("Fecha de la última visita de cardiología", "Fecha de la visita más reciente registrada en cardiología.", "Seguimiento", 5),
    "seguimiento_neuro": ("Seguimiento en neurología registrado", "Indica seguimiento registrado en neurología; por sí solo no establece un diagnóstico ni una enfermedad activa.", "Seguimiento", 5),
    "fecha_ultima_visita_neuro": ("Fecha de la última visita de neurología", "Fecha de la visita más reciente registrada en neurología.", "Seguimiento", 5),
    "seguimiento_cirugia_vascular": ("Seguimiento en cirugía vascular registrado", "Indica seguimiento registrado en cirugía vascular; por sí solo no establece un diagnóstico ni una enfermedad activa.", "Seguimiento", 5),
    "fecha_ultima_visita_cirugia_vascular": ("Fecha de la última visita de cirugía vascular", "Fecha de la visita más reciente registrada en cirugía vascular.", "Seguimiento", 5),
    "fecha_ultima_visita_especialidad": ("Fecha de la última visita de especialidad", "Fecha más reciente entre las visitas de especialidad incluidas por el sistema de origen; confirme qué especialidades se contabilizan.", "Seguimiento", 5),
    "tiene_especialidad_objetivo": ("Visita en una especialidad objetivo registrada", "Indica una visita en una especialidad objetivo; confirme qué especialidades se incluyen.", "Seguimiento", 5),
    "ecocardio_24m": ("Ecocardiograma en los últimos 24 meses", "Posiblemente indica un ecocardiograma reciente; confirme el período exacto y la regla del indicador.", "Pruebas de imagen", 6),
    "fecha_ultimo_ecocardio": ("Fecha del último ecocardiograma", "Fecha del ecocardiograma más reciente registrado; consulte el informe para conocer los detalles clínicos.", "Pruebas de imagen", 6),
    "angiotac_coronario_24m": ("Angio-TAC coronario en los últimos 24 meses", "Posiblemente indica un angio-TAC coronario reciente; confirme la regla exacta del indicador.", "Pruebas de imagen", 6),
    "fecha_ultimo_angiotac_coronario": ("Fecha del último angio-TAC coronario", "Fecha del angio-TAC coronario más reciente registrado.", "Pruebas de imagen", 6),
    "ecografia_vascular_24m": ("Ecografía vascular en los últimos 24 meses", "Posiblemente indica una ecografía vascular reciente; confirme la regla exacta del indicador.", "Pruebas de imagen", 6),
    "fecha_ultima_ecografia_vascular": ("Fecha de la última ecografía vascular", "Fecha de la ecografía vascular más reciente registrada.", "Pruebas de imagen", 6),
    "cumple_imagen_24m": ("Criterio de imagen en los últimos 24 meses", "Posible indicador combinado de pruebas de imagen recientes; documente la regla del sistema de origen antes de utilizarlo.", "Pruebas de imagen", 6),
    "tiene_ir_rx": ("Registro de IR/RX", "Indica un registro o una valoración de IR/RX. El significado de las siglas no está claro y debe confirmarse con el sistema de origen.", "Otros registros", 7),
    "fecha_ultimo_ir_rx": ("Fecha del último registro de IR/RX", "Fecha del registro más reciente de IR/RX; confirme el significado de estas siglas en el sistema de origen.", "Otros registros", 7),
    "tiene_ir_eco_atr": ("Registro de IR/eco_atr", "Indica un registro o una valoración de IR/eco_atr. Confirme las siglas y su significado con el sistema de origen.", "Otros registros", 7),
    "fecha_ultimo_ir_eco_atr": ("Fecha del último registro de IR/eco_atr", "Fecha del registro más reciente de IR/eco_atr; confirme el significado de estas siglas en el sistema de origen.", "Otros registros", 7),
}

missing_descriptions = set(df.columns) - column_descriptions.keys()
if missing_descriptions:
    raise ValueError(f"Add explanations for these source columns: {sorted(missing_descriptions)}")

column_dictionary = pd.DataFrame(
    [
        (column, *column_descriptions[column])
        for column in df.columns
    ],
    columns=["column_name", "readable_name", "meaning", "section", "section_order"],
)

info_columns_dict_path = Path("../public/info_columns_dict.csv")
column_dictionary.to_csv(info_columns_dict_path, index=False, encoding="utf-8-sig")
print(f"Formatted CSV saved to: {info_columns_dict_path}")



Formatted CSV saved to: data\info_columns_dict.csv


In [10]:
df["patient_ref"] = pd.factorize(df["patient_ref"])[0] + 1

formatted_sinids_csv_path = Path("./query_result_formatted_noids.csv")
df.to_csv(formatted_sinids_csv_path, index=False, encoding="utf-8-sig")
print(f"Formatted without ids CSV saved to: {formatted_sinids_csv_path}")


Formatted without ids CSV saved to: data\query_result_formatted_noids.csv
